# Jacobian determinant in the density push-forward formula

This notebook generates `fig:monge-jacobian-pushforward-density`.  It illustrates the change-of-variables formula

$$
\rho_\beta(T(x)) = \frac{\rho_\alpha(x)}{|\det DT(x)|},
$$

for a smooth local diffeomorphism.  The source density is locally uniform, so any nonuniformity in the target density is entirely caused by the Jacobian determinant.  The map uses a radially symmetric compression around the center, combined with a smooth radius-dependent twist and a small global rotation.  The displayed panels are cropped windows of a larger grid, so no artificial square boundary appears in the target panel; the target density forms a round central bump where the deformed grid cells are smallest.


In [1]:
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from PIL import Image
import subprocess
import tempfile

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import BLUE, RED, VIOLET, interp_color, figure_dir, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()

NAME = "monge-jacobian-pushforward-density"
OUT = figure_dir(NAME)
ARXIV_OUT = ROOT / "arxiv" / "figures"
THUMB_OUT = ROOT / "notebooks-figures" / "thumbnails"
ARXIV_OUT.mkdir(parents=True, exist_ok=True)
THUMB_OUT.mkdir(parents=True, exist_ok=True)


## A radial nonlinear diffeomorphism with analytic determinant

Let $z=x-(1/2,1/2)$, $r=|z|$, and define

$$
q(r)=1-b\exp\left(-\frac{r^2}{2\sigma^2}\right),\qquad
\varphi(r)=r q(r).
$$

For $0<b<1$, $q(r)>0$ and $\varphi$ is increasing, so the radial part is invertible.  We then add a radius-dependent twist

$$
\psi(r)=\theta_0+\omega\exp\left(-\frac{r^2}{2\tau^2}\right),
$$

and use

$$
T(x)=\frac12\mathbf 1+q(r) R_{\psi(r)}z.
$$

In polar coordinates this is $(r,\theta)\mapsto(\varphi(r),\theta+\psi(r))$.  The angular shear does not change area, hence

$$
\det DT(x)=q(r)\bigl(q(r)+r q'(r)\bigr),\qquad
q'(r)=\frac{b r}{\sigma^2}\exp\left(-\frac{r^2}{2\sigma^2}\right).
$$

The determinant is smallest at the center, so the pushed-forward density has a visibly isotropic central concentration while the grid lines reveal the nonlinear warp.  We draw a large source grid and crop the output window, which makes the deformation look unbounded rather than tied to the boundary of a square.


In [2]:
b = 0.70
sigma = 0.29
theta0 = np.deg2rad(8.0)
twist = np.deg2rad(26.0)
twist_sigma = 0.36
CENTER = np.array([0.5, 0.5])
RMAX_INVERSE = 2.6
VIEW_MIN, VIEW_MAX = -0.13, 1.13
GRID_MIN, GRID_MAX = -0.75, 1.75
GRID_STEP = 0.075


def q_of_r(r):
    return 1.0 - b * np.exp(-0.5 * (r / sigma) ** 2)


def qp_of_r(r):
    return b * (r / sigma**2) * np.exp(-0.5 * (r / sigma) ** 2)


def angle_of_r(r):
    return theta0 + twist * np.exp(-0.5 * (r / twist_sigma) ** 2)


def radial_quantities(x, y):
    u = x - 0.5
    v = y - 0.5
    r = np.hypot(u, v)
    q = q_of_r(r)
    ang = angle_of_r(r)
    return u, v, r, q, ang


def Txy(x, y):
    u, v, r, q, ang = radial_quantities(x, y)
    ca = np.cos(ang)
    sa = np.sin(ang)
    return 0.5 + q * (ca * u - sa * v), 0.5 + q * (sa * u + ca * v)


def det_jacobian(x, y):
    u = x - 0.5
    v = y - 0.5
    r = np.hypot(u, v)
    q = q_of_r(r)
    return q * (q + r * qp_of_r(r))

# Sanity check on a fine grid: the map is orientation preserving on a large window.
grid = np.linspace(GRID_MIN, GRID_MAX, 601)
X, Y = np.meshgrid(grid, grid, indexing="xy")
J = det_jacobian(X, Y)
print("determinant range", float(J.min()), float(J.max()))
assert J.min() > 0

# Finite-difference check of the analytic determinant.
rng = np.random.default_rng(0)
pts = rng.uniform(VIEW_MIN, VIEW_MAX, (20, 2))
h = 1e-6
fd_errors = []
for x, y in pts:
    txp = np.array(Txy(x + h, y))
    txm = np.array(Txy(x - h, y))
    typ = np.array(Txy(x, y + h))
    tym = np.array(Txy(x, y - h))
    Jfd = np.column_stack(((txp - txm) / (2 * h), (typ - tym) / (2 * h)))
    fd_errors.append(abs(np.linalg.det(Jfd) - det_jacobian(x, y)))
print("max finite-difference determinant error", float(max(fd_errors)))


determinant range 0.09000000000000002 1.1657097489823585
max finite-difference determinant error 4.1585423993240056e-10


## Inverting the map on a display grid

The inverse is still explicit up to a scalar monotone inversion.  The radius of a target point is $\varphi(r)$, so we recover $r$ by one-dimensional interpolation, undo the radius-dependent rotation, and divide by $q(r)$.  The target density is evaluated on the whole cropped display window, without masking by the image of a square.


In [3]:
r_grid = np.linspace(0, RMAX_INVERSE, 20001)
phi_grid = r_grid * q_of_r(r_grid)
assert np.all(np.diff(phi_grid) > 0)


def inverse_radius(rho):
    rho_clip = np.clip(rho, 0, phi_grid[-1])
    return np.interp(np.ravel(rho_clip), phi_grid, r_grid).reshape(np.shape(rho))


def inverse_map(Xt, Yt):
    w0 = Xt - 0.5
    w1 = Yt - 0.5
    rho = np.hypot(w0, w1)
    r = inverse_radius(rho)
    q = q_of_r(r)
    ang = angle_of_r(r)
    ca = np.cos(ang)
    sa = np.sin(ang)
    u = (ca * w0 + sa * w1) / q
    v = (-sa * w0 + ca * w1) / q
    x = u + 0.5
    y = v + 0.5
    inside = rho <= phi_grid[-1] + 1e-8
    return x, y, inside

# Fixed crop: the larger grid is clipped by the panel boundary.
xmin = ymin = VIEW_MIN
xmax = ymax = VIEW_MAX

N = 500
x_plot = np.linspace(xmin, xmax, N)
y_plot = np.linspace(ymin, ymax, N)
Xt, Yt = np.meshgrid(x_plot, y_plot, indexing="xy")
Xi, Yi, inside = inverse_map(Xt, Yt)
rho_raw = 1.0 / det_jacobian(Xi, Yi)
rho_beta = np.where(inside, rho_raw, np.nan)
dx = (xmax - xmin) / (N - 1)
dy = (ymax - ymin) / (N - 1)
mass_check = np.nansum(np.where(inside, rho_raw, 0.0)) * dx * dy
print(
    "target density range",
    float(np.nanmin(rho_beta)),
    float(np.nanmax(rho_beta)),
    "window integral",
    float(mass_check),
)


target density range 0.8578464670972739 11.08933156978287 window integral 1.6940785823264919


## Draw the source grid and the pushed-forward density

Both panels show cropped windows of a larger grid.  The source panel is locally uniform.  The target panel overlays the deformed grid on the pushed-forward density; the darkest region appears at the center, exactly where the radial compression makes the grid cells smallest.


In [4]:
import matplotlib.patheffects as pe


def blend(color, amount=0.55):
    rgb = np.array(to_rgb(color))
    return tuple((1 - amount) * rgb + amount * np.ones(3))


def grid_color(t):
    t = float(np.clip(t, 0, 1))
    if t <= 0.5:
        return interp_color(t / 0.5, RED, VIOLET)
    return interp_color((t - 0.5) / 0.5, VIOLET, BLUE)


def draw_grid(ax, transform=None, samples=900, alpha=0.88, lw=0.58):
    vals = np.arange(GRID_MIN, GRID_MAX + 0.5 * GRID_STEP, GRID_STEP)
    r = np.linspace(GRID_MIN, GRID_MAX, samples)
    denom = GRID_MAX - GRID_MIN
    for v in vals:
        col = grid_color((v - GRID_MIN) / denom)
        x = np.full_like(r, v)
        y = r.copy()
        if transform is not None:
            x, y = transform(x, y)
        ax.plot(x, y, color=col, lw=lw, alpha=alpha, zorder=6, solid_capstyle="round")
    for v in vals:
        col = blend(grid_color((v - GRID_MIN) / denom), 0.42)
        x = r.copy()
        y = np.full_like(r, v)
        if transform is not None:
            x, y = transform(x, y)
        ax.plot(x, y, color=col, lw=0.48 * lw / 0.58, alpha=0.72 * alpha, zorder=6, solid_capstyle="round")


def draw_source_panel(path):
    fig, ax = plt.subplots(figsize=(2.55, 2.55))
    remove_axes(ax)
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, facecolor=blend(RED, 0.92), edgecolor="none", zorder=0))
    draw_grid(ax, transform=None, samples=700, alpha=0.94, lw=0.58)
    save_pdf(fig, path, pad_inches=0.0)
    plt.close(fig)


def draw_target_panel(path):
    fig, ax = plt.subplots(figsize=(2.55, 2.55))
    remove_axes(ax)
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    cmap = LinearSegmentedColormap.from_list(
        "density_blue",
        ["#ffffff", "#eef4fb", "#bdd2ea", "#729bc8", BLUE],
    )
    vmax = np.nanquantile(rho_beta, 0.985)
    normalized = np.clip(np.nan_to_num(rho_beta, nan=0.0), 0, vmax) / vmax
    rgba = cmap(normalized)
    rgba[..., 3] = np.where(inside, 1.0, 0.0)
    ax.imshow(
        rgba,
        extent=[xmin, xmax, ymin, ymax],
        origin="lower",
        interpolation="bicubic",
        zorder=0,
    )
    draw_grid(ax, transform=Txy, samples=1300, alpha=0.72, lw=0.50)
    levels = np.nanquantile(rho_beta, np.linspace(0.54, 0.975, 8))
    contours = ax.contour(
        Xt,
        Yt,
        rho_beta,
        levels=np.unique(levels),
        colors="#1f2d3a",
        linewidths=0.46,
        alpha=0.64,
        zorder=8,
    )
    contours.set_path_effects([pe.Stroke(linewidth=0.80, foreground="white", alpha=0.30), pe.Normal()])
    save_pdf(fig, path, pad_inches=0.0)
    plt.close(fig)


def copy_to_arxiv(pdf_name):
    shutil.copyfile(OUT / pdf_name, ARXIV_OUT / f"{NAME}--{pdf_name}")


draw_source_panel(OUT / "source-grid.pdf")
draw_target_panel(OUT / "target-density-grid.pdf")
copy_to_arxiv("source-grid.pdf")
copy_to_arxiv("target-density-grid.pdf")


In [5]:
# Thumbnail for the notebook gallery.
def render_pdf_to_image(pdf, scale=3.2):
    try:
        import fitz
        doc = fitz.open(pdf)
        page = doc[0]
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        im = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        doc.close()
        return im
    except Exception:
        with tempfile.TemporaryDirectory() as tmp:
            prefix = Path(tmp) / "panel"
            subprocess.run(
                ["pdftoppm", "-png", "-r", str(int(72 * scale)), str(pdf), str(prefix)],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            return Image.open(f"{prefix}-1.png").convert("RGB")

imgs = [render_pdf_to_image(OUT / "source-grid.pdf"), render_pdf_to_image(OUT / "target-density-grid.pdf")]
h = max(im.height for im in imgs)
pad = 14
canvas = Image.new("RGB", (sum(im.width for im in imgs) + 3 * pad, h + 2 * pad), "white")
x0 = pad
for im in imgs:
    canvas.paste(im, (x0, pad + (h - im.height) // 2))
    x0 += im.width + pad
thumb = THUMB_OUT / f"{NAME}.png"
canvas.save(thumb, quality=94)
print(thumb)


/Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/monge-jacobian-pushforward-density.png
